# 01 - 数据准备

## 数据源
完整COT偏好数据已就绪：`train_preference_final_merged.json`（11955条）
- `chosen`: 正确COT解题步骤
- `rejected`: 错误COT解题步骤

## 本Notebook功能
1. 验证数据完整性
2. 从偏好数据自动派生SFT训练数据（chosen→cot）
3. 无需调用API，数据已完整

In [1]:
import os, sys, json

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("=== 调试信息 ===")
print(f"当前工作目录 (cwd): {os.getcwd()}")
print(f"当前文件: {__file__}" if '__file__' in locals() else "当前文件: 无法获取 (Jupyter 环境)")

# 最直接的方式：判断当前目录在哪里
cwd = os.getcwd()
workspace_root = None

if os.path.exists('/mnt/workspace/workspace/utils'):
    workspace_root = '/mnt/workspace/workspace'
elif os.path.exists('/mnt/workspace/utils'):
    workspace_root = '/mnt/workspace'
elif os.path.exists('../utils'):
    workspace_root = os.path.dirname(cwd)
elif os.path.exists('./utils'):
    workspace_root = cwd
elif 'workspace/notebooks' in cwd:
    workspace_root = os.path.dirname(cwd)
elif 'workspace' in cwd:
    workspace_root = cwd
else:
    workspace_root = cwd

sys.path.insert(0, workspace_root)
os.chdir(workspace_root)

print(f"\n设置后的工作目录: {os.getcwd()}")
print(f"sys.path 前两项: {sys.path[:2]}")
print(f"utils 目录是否存在: {os.path.exists(os.path.join(workspace_root, 'utils'))}")
print(f"utils 目录内容: {os.listdir(workspace_root) if os.path.exists(workspace_root) else '根目录不存在'}")

=== 调试信息 ===
当前工作目录 (cwd): e:\primary_math\workspace\notebooks
当前文件: 无法获取 (Jupyter 环境)

设置后的工作目录: e:\primary_math\workspace\notebooks
sys.path 前两项: ['e:\\primary_math\\workspace\\notebooks', 'c:\\Users\\ghf\\AppData\\Local\\Python\\pythoncore-3.12-64\\python312.zip']
utils 目录是否存在: False
utils 目录内容: ['01_Data_Preprocess.ipynb', '02_Train_Pipeline.ipynb', '03_Inference_Eval.ipynb']


In [2]:
from utils.pipeline_config import get_paths

paths = get_paths()

print(f"偏好数据: {paths['train_preference']}")
print(f"SFT数据: {paths['train_cot']}")
print(f"数据目录: {paths['data_dir']}")

ModuleNotFoundError: No module named 'utils'

## ===== 验证数据完整性 =====

In [ ]:
pref_path = paths['train_preference']
with open(pref_path, 'r', encoding='utf-8') as f:
    pref_data = json.load(f)

print(f'偏好数据: {len(pref_data)} 条')
print(f'字段: {list(pref_data[0].keys())}')

no_chosen = sum(1 for x in pref_data if not x.get('chosen','').strip())
no_rejected = sum(1 for x in pref_data if not x.get('rejected','').strip())
print(f'空chosen: {no_chosen}, 空rejected: {no_rejected}')

print(f'\n样本:')
s = pref_data[0]
print(f'  问题: {s["question"][:60]}...')
print(f'  答案: {s["answer"]}')
print(f'  chosen: {s["chosen"][:80]}...')
print(f'  rejected: {s["rejected"][:80]}...')

In [ ]:
sft_path = paths['train_cot']
with open(sft_path, 'r', encoding='utf-8') as f:
    sft_data = json.load(f)

print(f'SFT数据: {len(sft_data)} 条')
has_cot = sum(1 for x in sft_data if x.get('cot','').strip())
print(f'有COT: {has_cot}/{len(sft_data)}')

print(f'\n样本:')
s = sft_data[0]
print(f'  问题: {s["question"][:60]}...')
print(f'  COT: {s["cot"][:80]}...')
print(f'  答案: {s["answer"]}')

## ===== 数据就绪 =====

数据已完整，可直接进入训练流水线（02_Train_Pipeline.ipynb）

In [ ]:
print('='*50)
print('数据准备完成')
print('='*50)
print(f'  SFT数据: {sft_path} ({len(sft_data)}条)')
print(f'  DPO数据: {pref_path} ({len(pref_data)}条)')
print(f'  GRPO数据: {sft_path} ({len(sft_data)}条)')
print()
print('下一步: 运行 02_Train_Pipeline.ipynb')